# F6 homework: racing optimisers

In this notebook you implement four optimisers yourself (gradient descent, mini-batch SGD, momentum
and Adam) and race them on two problems:

- **a linear regression** with 2,000 examples and 5 features, where we know the exact answer from
  `np.linalg.lstsq`;
- **the Rosenbrock function** $f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$, a classic test problem with a
  long, curved, narrow valley and its minimum at $(1, 1)$.

You will plot loss curves and optimisation paths, and run a learning-rate sweep to see how sensitive
each method is.

**How to work:** fill in every cell marked `# TODO`, then run the check cell below it. A check cell
prints `OK` when your code is right. Only NumPy and matplotlib are needed; everything runs in well
under a minute on any laptop or on free Colab (CPU).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## The two problems (given)

The regression features are on very different scales, so we standardise them (lesson 5) and append a
column of ones for the bias (lesson 6). `w_star` is the exact least-squares solution and `L_star` its
loss: no optimiser can do better.

In [ ]:
rng = np.random.default_rng(0)
n, d = 2000, 5
X = rng.normal(size=(n, d)) * [1, 5, 0.2, 50, 2] + [0, 10, 1, 300, -4]
w_true = np.array([1.5, -0.4, 8.0, 0.03, 2.0])
y = X @ w_true + 7 + rng.normal(0, 1.0, n)

Z = (X - X.mean(axis=0)) / X.std(axis=0)
A = np.column_stack([Z, np.ones(n)])

def reg_loss(w):
    return np.mean((A @ w - y) ** 2)

w_star, *_ = np.linalg.lstsq(A, y)
L_star = reg_loss(w_star)
print(f"best possible loss L_star = {L_star:.4f}")

def rosen(p):
    x, y_ = p
    return (1 - x) ** 2 + 100 * (y_ - x ** 2) ** 2

P0 = np.array([-1.5, 2.0])            # starting point for every Rosenbrock run
print("Rosenbrock at the start:", rosen(P0), " at (1, 1):", rosen(np.array([1.0, 1.0])))

## Task 1: the gradients

1. `reg_grad(w, Ab, yb)`: the gradient of the MSE on a batch, $\frac{2}{|B|} A_B^\top (A_B\mathbf{w} - \mathbf{y}_B)$,
   where `Ab` holds some rows of `A` and `yb` the matching targets. With all rows it is the full gradient.
2. `rosen_grad(p)`: the gradient of the Rosenbrock function. Derive both partial derivatives **on paper**
   with the chain rule first. (Hint: the derivative of $100(y - x^2)^2$ with respect to $x$ has a factor
   $-2x$ from the inside.)

The check compares both with central differences (F4, lesson 5 of F6).

In [ ]:
def reg_grad(w, Ab, yb):
    # TODO: return the mean squared error gradient on this batch
    pass

def rosen_grad(p):
    x, y_ = p
    # TODO: return np.array([df/dx, df/dy])
    pass

In [ ]:
def _num_grad(f, p, h=1e-6):
    p = np.asarray(p, dtype=float)
    return np.array([(f(p + e) - f(p - e)) / (2 * h) for e in np.eye(len(p)) * h])

def _rel(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b))

_r = np.random.default_rng(1)
for _ in range(3):
    _w = _r.normal(size=d + 1)
    assert _rel(reg_grad(_w, A, y), _num_grad(reg_loss, _w)) < 1e-6, "reg_grad fails the gradient check"
    _p = _r.uniform(-2, 2, size=2)
    assert _rel(rosen_grad(_p), _num_grad(rosen, _p)) < 1e-6, f"rosen_grad fails the gradient check at {_p}"
_idx = np.arange(10)
assert np.allclose(reg_grad(w_star, A, y), 0, atol=1e-8), "The full gradient should vanish at w_star"
assert reg_grad(np.zeros(d + 1), A[_idx], y[_idx]).shape == (d + 1,), "reg_grad on a batch should return one entry per weight"
assert np.allclose(rosen_grad(np.array([1.0, 1.0])), 0), "The Rosenbrock gradient is zero at (1, 1)"
print("OK")

## Task 2: three optimiser steps

Every optimiser has the same interface:

```python
new_p = step(p, g, state, lr, **hyperparameters)
```

- `p` is the current parameter vector and `g` the gradient there;
- `state` is a dictionary that starts **empty** (`{}`) and that the step may use to remember things
  between calls (a velocity, moving averages, a step counter). Store what you need in it, for example
  `state["v"] = v`;
- the function returns the new parameters.

Implement:

1. `gd_step(p, g, state, lr)`: plain gradient descent (it ignores `state`). Mini-batch SGD is this same
   step fed with mini-batch gradients.
2. `momentum_step(p, g, state, lr, beta=0.9)`: $\mathbf{v} \leftarrow \beta\mathbf{v} + \mathbf{g}$,
   $\mathbf{p} \leftarrow \mathbf{p} - \eta\mathbf{v}$, with $\mathbf{v}$ starting at zero.
3. `adam_step(p, g, state, lr, b1=0.9, b2=0.999, eps=1e-8)`: Adam with bias correction, keeping the step
   counter `t` (starting at 1 on the first call), `m` and `v` in `state`.

In [ ]:
def gd_step(p, g, state, lr):
    # TODO
    pass

def momentum_step(p, g, state, lr, beta=0.9):
    # TODO: read the velocity from state (zeros the first time), update it, store it back
    pass

def adam_step(p, g, state, lr, b1=0.9, b2=0.999, eps=1e-8):
    # TODO: keep t, m and v in state; remember the bias correction
    pass

In [ ]:
_p = np.array([3.0, 1.0]); _g = np.array([6.0, 100.0])
assert np.allclose(gd_step(_p, _g, {}, 0.01), [2.94, 0.0]), "gd_step: p - lr * g"
_s = {}
_p1 = momentum_step(_p, _g, _s, 0.01)
assert np.allclose(_p1, [2.94, 0.0]), "The first momentum step (v = 0) is a plain gradient step"
_p2 = momentum_step(_p1, _g, _s, 0.01)
assert np.allclose(_p2, _p1 - 0.01 * (0.9 * _g + _g)), "The second momentum step should use v = 0.9 * g + g"
assert np.allclose(_p, [3.0, 1.0]), "Do not modify p in place"
_s = {}
_q = adam_step(_p, _g, _s, 0.1)
assert np.allclose(_q, [2.9, 0.9], atol=1e-6), "Adam's first step moves every parameter by lr: check the bias correction"
_ref_m, _ref_v, _ref_p = np.zeros(2), np.zeros(2), _q.copy()
_ref_m = 0.1 * _g; _ref_v = 0.001 * _g ** 2
for _t in [2, 3]:
    _gg = np.array([1.0, -2.0]) * _t
    _ref_m = 0.9 * _ref_m + 0.1 * _gg; _ref_v = 0.999 * _ref_v + 0.001 * _gg ** 2
    _ref_p = _ref_p - 0.1 * (_ref_m / (1 - 0.9 ** _t)) / (np.sqrt(_ref_v / (1 - 0.999 ** _t)) + 1e-8)
    _q = adam_step(_q, _gg, _s, 0.1)
assert np.allclose(_q, _ref_p), "After three Adam steps your parameters differ from the reference"
print("OK")

## Task 3: race on the regression

Write `train_regression(step, lr, epochs, batch_size, seed=0, **hyper)` that:

1. starts from `w = np.zeros(d + 1)` and an empty `state = {}`, and makes a generator
   `np.random.default_rng(seed)`;
2. for each epoch, shuffles with `rng.permutation(n)`, cuts the order into batches of `batch_size`, and for
   each batch computes `reg_grad` on that batch and calls `w = step(w, g, state, lr, **hyper)`;
3. records `reg_loss(w)` on the **full** data once before training and after every epoch;
4. returns `(w, losses)` with `losses` a NumPy array of length `epochs + 1`.

Full-batch gradient descent is just `batch_size=n` (one step per epoch).

In [ ]:
def train_regression(step, lr, epochs, batch_size, seed=0, **hyper):
    # TODO
    pass

In [ ]:
EPOCHS = 20
runs = {
    "GD (full batch, lr 0.5)":  train_regression(gd_step, 0.5, EPOCHS, n),
    "SGD (B=32, lr 0.01)":      train_regression(gd_step, 0.01, EPOCHS, 32),
    "Momentum (B=32, lr 0.001)": train_regression(momentum_step, 0.001, EPOCHS, 32, beta=0.9),
    "Adam (B=32, lr 0.05)":     train_regression(adam_step, 0.05, EPOCHS, 32),
}
for name, (w, losses) in runs.items():
    assert len(losses) == EPOCHS + 1, f"{name}: losses should have epochs + 1 entries"
    assert np.isclose(losses[0], reg_loss(np.zeros(d + 1))), "losses[0] should be the loss before training"
    print(f"{name:27s} final loss {losses[-1]:.4f}   max |w - w_star| {np.max(np.abs(w - w_star)):.3f}")
    assert losses[-1] < 1.02 * L_star, f"{name} should end within 2% of L_star = {L_star:.4f}"
print("OK")

Now plot the **excess loss** $L - L^\star$ against the epoch for all four runs on one figure, with a
logarithmic y-axis (`plt.yscale("log")`), a legend and labelled axes. The excess loss shows how far each
method is from the best possible fit; on a log scale steady convergence looks like a straight line.

In [ ]:
plt.figure(figsize=(7, 4))
# TODO: one line per run in `runs`, plotting losses - L_star against the epoch number
plt.yscale("log")
# TODO: axis labels, title and legend
plt.show()

**Write 3 to 5 sentences** (edit this cell): Per epoch, which method gets close fastest, and why does
full-batch GD need a large learning rate to compete? Which curves stop improving and wobble, and what
from lesson 3 explains that? Try Adam with `lr=0.01`: why is it so slow here? (Hint: how far does Adam
move each parameter per step, and how far is `w_star` from zero?)

## Task 4: race on the Rosenbrock valley

Write `run_2d(step, p0, lr, steps, **hyper)` that runs an optimiser on the Rosenbrock function from
`p0` with full gradients from `rosen_grad`, and returns the **history**: an array of shape
`(k + 1, 2)` holding the start and every point visited. If the loss becomes non-finite or larger than
`1e10`, stop early and return the history so far (including that last point).

In [ ]:
def run_2d(step, p0, lr, steps, **hyper):
    # TODO
    pass

In [ ]:
STEPS = 3000
paths = {
    "GD (lr 1.5e-3)":           run_2d(gd_step, P0, 1.5e-3, STEPS),
    "Momentum (lr 4e-4, β 0.9)": run_2d(momentum_step, P0, 4e-4, STEPS, beta=0.9),
    "Adam (lr 0.02)":           run_2d(adam_step, P0, 0.02, STEPS),
}
for name, h in paths.items():
    assert h.shape == (STEPS + 1, 2), f"{name}: expected history shape {(STEPS + 1, 2)}, got {h.shape}"
    assert np.allclose(h[0], P0), "Row 0 of the history must be the start"
    print(f"{name:26s} ends at {np.round(h[-1], 4)}   f = {rosen(h[-1]):.2e}")
assert rosen(paths["GD (lr 1.5e-3)"][-1]) < 0.05, "GD should get well down the valley"
assert rosen(paths["Momentum (lr 4e-4, β 0.9)"][-1]) < 1e-3, "Momentum should get close to (1, 1)"
assert rosen(paths["Adam (lr 0.02)"][-1]) < 1e-6, "Adam should reach (1, 1) almost exactly"
_h = run_2d(gd_step, P0, 3e-3, STEPS)
assert len(_h) < STEPS + 1 and not (rosen(_h[-1]) < 1e10), "GD with lr 3e-3 diverges: run_2d should stop early"
print("OK")

Draw the paths on top of the contours. Rosenbrock values span many orders of magnitude, so use
logarithmically spaced contour levels. Mark the start and the minimum $(1, 1)$, and give the figure a
legend and labelled axes.

In [ ]:
xs = np.linspace(-2, 2, 300)
ys = np.linspace(-1, 3, 300)
XX, YY = np.meshgrid(xs, ys)
FF = (1 - XX) ** 2 + 100 * (YY - XX ** 2) ** 2

plt.figure(figsize=(7, 6))
plt.contour(XX, YY, FF, levels=np.logspace(-1, 3.5, 14), cmap="Greys", linewidths=0.8)
# TODO: plot each path in `paths` (x = column 0, y = column 1) with a label
# TODO: mark the start P0 and the minimum (1, 1)
# TODO: axis labels, title and legend
plt.show()

Also plot $f$ against the step number for the three runs, with a log y-axis (`plt.semilogy`).

In [ ]:
plt.figure(figsize=(7, 4))
# TODO
plt.show()

## Task 5: learning-rate sensitivity

Write `sweep(step, lrs, steps=3000, **hyper)` that returns a list with, for each learning rate, the final
Rosenbrock value, or `np.inf` if the run stopped early because it diverged. Then print a table for the
three optimisers with the grids below.

In [ ]:
def sweep(step, lrs, steps=3000, **hyper):
    # TODO
    pass

In [ ]:
grids = {
    "GD":       (gd_step,       [5e-4, 1e-3, 1.5e-3, 2e-3, 2.5e-3, 3e-3], {}),
    "Momentum": (momentum_step, [1e-4, 2e-4, 4e-4, 1e-3, 2e-3, 4e-3], {"beta": 0.9}),
    "Adam":     (adam_step,     [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1], {}),
}
results = {}
for name, (step, lrs, hyper) in grids.items():
    results[name] = sweep(step, lrs, **hyper)
    print(name)
    for lr, f in zip(lrs, results[name]):
        print(f"   lr {lr:<8g} final f {f:.2e}")
assert np.isinf(results["GD"][-1]), "GD with lr 3e-3 should be reported as diverged (inf)"
assert all(np.isfinite(results["Adam"])), "Adam should not diverge anywhere on its grid"
assert len(results["Momentum"]) == 6, "sweep should return one value per learning rate"
print("OK")

**Write 4 to 6 sentences** (edit this cell). Note that the learning rates used in the Task 4 race were
not tuned optima: your sweep may show, for example, that momentum at `1e-3` beats the `4e-4` used there.

- What is the largest stable learning rate for GD, and how does it relate to the curvature of the valley
  (lesson 2)?
- Compare momentum's best learning rate with GD's. Using $1/(1 - \beta)$, explain why momentum can make
  faster progress along the valley floor without diverging across it. Where does momentum finally
  diverge? (For heavy-ball momentum on a quadratic the limit is about $\eta < 2(1 + \beta)/\lambda$, where
  $\lambda$ is the largest curvature, compared with $2/\lambda$ for GD.)
- Over how many factors of 10 does Adam's learning rate still give a usable result? Why does dividing
  by $\sqrt{\hat{\mathbf{v}}}$ make it less sensitive?
- Based on both problems, which optimiser would you try first on a new problem, and what would you tune
  first?

### Stretch (optional)

1. Add `rmsprop_step` and race it too.
2. Give SGD on the regression a cosine learning-rate schedule (lesson 5) and show the final wobble shrinks.
3. Start Rosenbrock from a few other points; does the ranking of the optimisers change?